# 文献から知識グラフを作り、Graph RAG で質問に答える

材料科学の論文では、成膜条件、作製した試料、測定した物性が、実験方法や結果の節に分かれて記載されています。たとえば「200 ℃で成膜した ZnO 試料の移動度はどのくらいか？」に答えるには、温度と移動度を取り出し、**両方が同じ試料に対応していること**を確認する必要があります。知識グラフでは、この対応をノードと関係で表します。

## 1. スキーマを与えて文献から知識グラフを作る

まず、答えたい質問に合わせて、何を抽出し、どう結び付けるかを**スキーマ**で定めます。成膜条件と移動度を比較したい場合は、次のように設計できます。

| ノード型 | 記録する情報 | 他のノードとの関係 |
|---|---|---|
| 成膜プロセス | 成膜法、温度、前駆体 | 試料を作製した |
| 試料 | 試料ID、材料 | 測定値を持つ |
| 測定値 | 物性名、数値、単位 | 論文に報告された |
| 論文 | DOI、該当節、原文の根拠箇所 | 成膜条件や測定値の出典になる |

次の一節は、抽出の流れを示すために作った架空の文章です。

> ZnO 薄膜 S1 を 200 ℃で原子層堆積（ALD）により作製した。S1 の電子移動度は 12 cm²/(V·s) であった。

この文章とスキーマを LLM に渡し、成膜プロセス、試料 S1、移動度の測定値を抽出します。そのうえで、`成膜プロセス → 作製した → S1` と `S1 → 測定値を持つ → 電子移動度 12 cm²/(V·s)` の関係を記録します。本文に前駆体の記載がなければ、その項目は空欄にします。値とともに原文の根拠箇所を保存しておくと、値や単位の読み違い、試料の取り違えを後から確認できます。

実際に ALD 論文にこの考え方を適用した研究が **SciKGExtract** です。Sadruddin と D’Souza は、専門家が検証したスキーマに従って ZnO と IGZO の論文から実験情報を抽出し、化学名の正規化や抽出結果の検証を経て知識グラフへ統合しました。また、Dunn らは、材料化学の論文要旨から、母材とドーパントの関係や材料の組成・相・形態などを、用途ごとのスキーマに沿った JSON として出力する方法を示しました。 [Sadruddin & D’Souza, *Agentic schema-guided extraction of materials process knowledge from scientific literature*, arXiv:2610.06322（2026）](https://arxiv.org/abs/2610.06322) · [Dunn et al., *Structured information extraction from complex scientific text with fine-tuned large language models*, arXiv:2212.05238（2022）](https://arxiv.org/abs/2212.05238)

## 2. 知識グラフを検索する

知識グラフにデータを登録すると、**「200 ℃で ALD 成膜した ZnO 試料について、電子移動度と出典を取得する」**という条件をクエリで表せます。使う検索言語は、グラフの表現やデータベースによって異なります。

**RDF と SPARQL。** RDF では、「S1 は測定値 M1 を持つ」のような事実を、主語・述語・目的語の三つ組で表します。オントロジーで定義したクラスや関係を RDF データに使えば、SPARQL の検索条件にも同じ語彙を使えます。たとえば、「成膜プロセス → 試料 → 測定値 → 出典」というつながりを検索パターンとして指定します。 [W3C, *SPARQL 1.1 Query Language*（2013）](https://www.w3.org/TR/sparql11-query/)

**Neo4j と Cypher。** Neo4j は、ノードと関係にプロパティを持たせられるグラフデータベースです。たとえば成膜プロセスのノードに `temperature_c: 200` を持たせ、試料ノードと `PRODUCED` で結び付けます。Cypher の `MATCH` で、条件に合うノードと関係のパターンを探し、測定値と出典を取り出します。 [Neo4j, *Cypher Manual — MATCH*](https://neo4j.com/docs/cypher-manual/current/clauses/match/)

| グラフの表現・データベース | 検索言語 | 検索の基本 |
|---|---|---|
| RDF グラフ | SPARQL | 主語・述語・目的語のパターンを照合する |
| Neo4j のプロパティグラフ | Cypher | ノード・関係・プロパティのパターンを照合する |

## 3. 検索結果を使って Graph RAG で回答する

Graph RAG では、自然言語の質問を受け取ると、関連するノードや関係をグラフから取得し、その検索結果を根拠として LLM が回答を生成します。この例では、SPARQL や Cypher が**該当する試料・成膜条件・測定値を取得する処理**を担い、LLM が**取得した事実を質問に沿って説明する処理**を担います。

たとえば「200 ℃で ALD 成膜した ZnO の電子移動度は？」と質問されたら、まず条件に合う成膜プロセスを探し、そこから試料、測定値、出典をたどります。LLM は得られた記録を使って「試料 S1 では、電子移動度は 12 cm²/(V·s) でした」と答え、該当する論文の箇所を示します。複数の試料が該当する場合は、それぞれの条件と測定値を対応づけて提示します。一つの試料についての報告から、200 ℃が移動度を高めるという因果関係までは導けません。

この例は、試料と測定値のような具体的な関係をたどる Graph RAG です。グラフを使った検索には、文書全体にまたがる質問に答える方法もあります。Edge らの研究では、文書からエンティティのグラフを構築し、関連の強いグループごとに要約を作り、それらを使って文書集合全体についての質問に答えています。 [Edge et al., *From Local to Global: A Graph RAG Approach to Query-Focused Summarization*, arXiv:2404.16130（2024）](https://arxiv.org/abs/2404.16130)

## ベクトル検索型 RAG から Graph RAG へ

ベクトル検索型 RAG では、質問と意味が近い文書チャンクを取得し、回答生成に使います。埋め込み、データベース、文書登録と検索の手順は [RAG の基礎](8300.0150.RAG.ipynb) を参照してください。以下では、複数箇所に分かれた情報を関係で結び付ける場合を考えます。

## 従来RAGの限界

RAGが検索する対象は通常、

```text
文章
段落
チャンク
```

です。

そのため、

```text
段落1
NH3 plasmaを用いた。

段落2
窒素供給量が増加した。

段落3
TiNの抵抗率が低下した。
```

という情報が論文中の離れた場所に存在すると、

```text
NH3 plasma
 ↓
窒素供給量増加
 ↓
抵抗率低下
```

という因果関係を見つけることが難しくなります。

つまり、Embedding検索は

> 「意味が近い文章を探す」

ことは得意ですが、

> 「複数の知識をつなげて因果関係を辿る」

ことはあまり得意ではありません。

この問題を解決するために登場したのが **Graph RAG** です。

---



# 2. Context Graphとは？

Context Graphとは、

> **文書の内容をNode（Entity）とEdge（関係）で表現した知識グラフ**

です。

通常のRAGでは文章やチャンクを検索しますが、Context Graphでは文書中の重要な概念とその関係をグラフとして保存します。

---



## NodeとEdge

Context Graphは、

* **Node（Entity）**：対象となる概念
* **Edge（Relation）**：Node同士の関係

から構成されます。

例えば論文中に

```text
NH3 plasmaを使用した。

窒素供給量が増加した。

TiNの抵抗率が低下した。
```

と書かれていたとします。

これをグラフ化すると、

```text
[NH3 plasma]
      |
      v
[N供給量増加]
      |
      v
[抵抗率低下]
```

のようになります。

---



## なぜGraphにするのか？

論文では関連する情報が離れた場所に書かれることがよくあります。

例えば、

```text
第2章
NH3 plasmaを使用した。

第4章
窒素供給量が増加した。

第6章
抵抗率が低下した。
```

のような場合、人間は

```text
NH3 plasma
 ↓
窒素供給量増加
 ↓
抵抗率低下
```

という因果関係を理解できます。

しかし通常のRAGは文章単位で検索するため、このような長距離の関係を見つけることが苦手です。

Context Graphでは関係をEdgeとして保存するため、複数の知識をつなげて辿ることができます。

---



## Context Graph設計の重要性

Context Graphを作る際に最も重要なのは、

> **どのようなNode（Entity）とEdge（Relation）を定義するか**

です。

実はGraph RAGの性能は、

> Graph検索アルゴリズム

よりも、

> Graphそのものの設計

に大きく依存します。

---



## 悪い例：論文をそのままGraph化する

論文には様々な概念が登場します。

```text
材料
前駆体
温度
圧力
プラズマ
測定方法
結晶構造
欠陥
キャリア濃度
...
```

これらをすべて別種類のNodeとして作ると、

```text
TemperatureNode
PressureNode
PrecursorNode
AnnealingNode
CarrierDensityNode
MobilityNode
...
```

のようにNode種類が増え続けます。

結果として、

```text
Node種類 = 数十〜数百
Edge種類 = 数十〜数百
```

となり、

* Schemaが複雑になる
* 抽出が不安定になる
* 論文ごとに異なるGraphになる
* 検索が難しくなる

という問題が発生します。

---



## 良い例：質問に答えるためのGraph

Context Graphの目的は、

> 論文を完全に再現すること

ではありません。

目的は、

> ユーザーの質問に答えられるように知識を整理すること

です。

例えばALD論文のGraph RAGなら、

```text
Strategy
↓
Process
↓
MaterialSystem
↓
Device
↓
Property
```

という少数のNode型に統一します。

詳細な違いはNode属性として保存します。

例えば、

```text
Process
 ├ temperature = 200℃
 ├ pressure = 1 Torr
 └ precursor = TiCl4
```

のように表現します。

---



## Node種類を減らす理由

Node種類が少ないほど、

```text
論文A
論文B
論文C
```

を同じ構造で表現できます。

例えば、

```text
Process
↓
Material
↓
Property
```

という共通構造があれば、

異なる論文間でも比較や統合が可能になります。

その結果、

```text
論文A
+
論文B
+
論文C
```

を横断した検索ができるようになります。

---



## Context Graphのイメージ

材料プロセス分野では、

```text
(Process)
ALD temperature
        |
        v
(Material)
TiN film
        |
        v
(Property)
Resistivity
```

のようなGraphになります。

より一般的には、

```text
Strategy
↓
Process
↓
MaterialSystem
↓
Device
↓
Property
```

という因果の流れで知識を整理します。

---



## Context Graphの目的

Context Graphは、

> 「論文を説明するためのGraph」

ではなく、

> 「質問に答えるためのGraph」

です。

そのため、

* Node種類はできるだけ少なくする
* Edge種類も整理する
* 論文ごとに異なる表現を避ける
* Multi-hop検索に必要な因果関係を優先して残す

ことが重要です。

つまりContext Graphとは、

> 文書を検索しやすい知識ネットワークへ変換したもの

であり、その品質はNodeとEdgeの設計によって大きく左右されます。



## 想定質問とContext Graph設計

Graph RAGで利用するContext Graphの目的は、

> **ユーザーの質問に答えられるように知識を整理すること**

です。

そのため、本来は

```text
どのような質問に答えたいか
```

を先に考え、その質問に必要なNode（Entity）とEdge（Relation）を設計するのが望ましいです。

例えば材料プロセス分野では、

```text
ALD温度は物性にどう影響するか？

どの材料が高移動度を示すか？

なぜ抵抗率が低下したか？
```

といった想定質問からSchemaを設計します。

このような設計では、

```text
質問
↓
必要な知識
↓
Node/Edge設計
```

という流れになります。

---



## 質問が無い場合のContext Graph設計

一方で、

> 「どのような質問が来るか分からないので、なるべく多くの情報を保持したい」

という考え方もあります。

この場合は、論文中の様々な概念を表現できるようにEntityやEdgeを定義します。

```text
論文
↓
知識抽出
↓
Entity/Edge設計
↓
Context Graph
```

という流れになります。

しかし、この方法では

* Entity種類が増える
* Edge種類が増える
* Schema設計が複雑になる
* 抽出精度が低下しやすい
* Graph検索が難しくなる

という問題が発生します。

そのため、

```text
論文を完全に表現するGraph
```

を作ろうとすると、NodeやEdgeの種類が爆発しやすくなります。

---



## Question-driven設計と Literature-driven設計

Context Graph設計には大きく二つの考え方があります。

### Question-driven設計

まず想定質問を定め、その質問に答えるためのSchemaを設計する方法です。

```text
想定質問
↓
Schema設計
↓
Context Graph
↓
Graph RAG
```

例：

```text
ALD温度は物性にどう影響するか？
```

↓

```text
Process
↓
Material
↓
Property
```

というSchemaを設計する。

---

### Literature-driven設計

想定質問を定めず、論文から共通する概念を抽出してSchemaを設計する方法です。

```text
論文群
↓
共通概念抽出
↓
Schema設計
↓
Context Graph
```

この方法では将来の様々な質問に対応できる可能性がありますが、

* Entity設計が難しい
* Edge設計が難しい
* 過剰に複雑なSchemaになりやすい

という課題があります。

---



## 実際のGraph RAG開発

実際には、

```text
Question-driven
```

と

```text
Literature-driven
```

の中間を目指すことが多くなります。

つまり、

* 想定質問を意識する
* Node種類はなるべく少なくする
* 多くの論文で共通に使えるSchemaにする

という方針です。

Context Graph設計において最も重要なのは、

> **論文を完全に再現することではなく、質問に答えられる形で知識を整理すること**

です。
---



# 3. Graph RAGとは？

Graph RAG（Graph Retrieval-Augmented Generation）は、

> **Context Graphを検索して、その結果をLLMへ与えて回答を生成する技術**

です。

RAGとContext Graphを組み合わせた仕組みと考えることができます。

---



## RAGとGraph RAGの違い

通常のRAGでは、

```text
論文
↓
チャンク分割
↓
Embedding
↓
Vector Database
↓
検索
↓
LLM
```

という流れになります。

検索対象は

```text
文章
段落
チャンク
```

です。

一方、Graph RAGでは、

```text
論文
↓
Context Graph
↓
Graph Database
↓
Graph検索
↓
LLM
```

という流れになります。

検索対象は

```text
Node
Edge
Path
```

です。

---



## 質問はGraph検索へ変換される

Graph RAGの最も重要な特徴は、

> **質問がGraph Databaseに対する検索問題へ変換されること**

です。

通常のRAGでは、

```text
質問
↓
Embedding生成
↓
Vector Database検索
↓
関連チャンク取得
↓
LLM
```

という処理を行います。

つまり、

> 「意味的に近い文章を探す」

ことが目的です。

一方、Graph RAGでは、

```text
質問
↓
Graph Query生成
↓
Graph Database検索
↓
関連Node・Edge・Path取得
↓
LLM
```

という処理になります。

つまり、

> 「関連する知識の経路（Path）を探す」

ことが目的になります。

---



## 例：通常RAG

質問

```text
ALD温度は移動度にどう影響するか？
```

通常のRAGでは、

```text
ALD temperature
mobility
```

を含むチャンクが検索されます。

```text
質問
↓
Embedding
↓
類似チャンク検索
```

となります。

---



## 例：Graph RAG

同じ質問に対して、

```text
ALD温度は移動度にどう影響するか？
```

Graph RAGは質問をGraph上の探索問題として解釈します。

例えばGraph内に

```text
(Process)
ALD temperature

(Property)
Mobility
```

というNodeがあるとすると、

探索したいのは

```text
Process
↓
?
↓
Property
```

という経路です。

Graph Queryとしては、

```text
ALD temperature
から
Mobility
へ到達するPathを探せ
```

という問題になります。

---



## Graph Databaseの検索

Graph Databaseには、

```text
Strategy
↓
Process
↓
MaterialSystem
↓
Device
↓
Property
```

のような知識ネットワークが保存されています。

例えば、

```text
ALD temperature
↓
Film density
↓
Carrier concentration
↓
Mobility
```

という経路が存在すれば、

Graph検索によってこのPathを取得できます。

つまりGraph RAGでは、

```text
質問
↓
Graph Query
↓
Graph探索
↓
Evidence Path取得
```

が行われます。

---



## Graph Retrieval

Graph RAGは質問に応じてGraphを探索します。

例えば、

```text
TiNの低抵抗化には何が有効か？
```

という質問に対して、

```text
Property
↓
Material
↓
Process
```

を逆向きに辿ることができます。

取得されるのは文章ではなく、

```text
Node
Edge
Path
```

です。

---



## Multi-hop検索

Graph RAG最大の特徴は、

> **複数段階の関係（multi-hop）を辿れること**

です。

通常のRAGは、

```text
質問
↓
意味が近いチャンク
```

を探します。

一方Graph RAGは、

```text
質問
↓
Node
↓
Edge
↓
Node
↓
Edge
↓
Node
```

という探索ができます。

---



## 例1：因果関係の探索

質問：

```text
なぜTiNの抵抗率が低下したのか？
```

Graph内に

```text
NH3 plasma
↓
N供給量増加
↓
Vacancy減少
↓
抵抗率低下
```

が存在するとします。

Graph RAGはこの経路を取得できます。

```text
NH3 plasma
→ N供給量増加
→ Vacancy減少
→ 抵抗率低下
```

そしてLLMは、

> NH3 plasmaによって窒素供給量が増加し、欠陥濃度が低下した結果、抵抗率が低下した。

という説明を生成できます。

---



## 例2：論文横断検索

Graph RAGでは複数論文を統合できます。

例えば、

論文A

```text
ALD温度
↓
膜密度
```

論文B

```text
膜密度
↓
移動度
```

論文C

```text
移動度
↓
デバイス性能
```

があったとします。

通常のRAGでは、

これらの関係が異なる論文に分散しているため発見が困難です。

しかしGraph RAGでは、

```text
ALD温度
↓
膜密度
↓
移動度
↓
デバイス性能
```

という知識パスを構築できます。

---



## なぜContext Graphが重要なのか？

Graph RAGの性能は、

> 検索アルゴリズム

よりも

> Context Graph設計

に大きく依存します。

Graph検索は、

```text
Node
+
Edge
+
Path
```

しか利用できません。

そのため、

必要なEntityやRelationがGraphに存在しなければ検索できません。

---



## Question-driven Graph RAG

Graph RAGでは、

まず

```text
どのような質問に答えたいか
```

を考えることが重要です。

例えば、

```text
ALD温度は物性にどう影響するか？

どの材料が高移動度を示すか？

なぜ抵抗率が低下したか？
```

という質問を想定するなら、

```text
Strategy
↓
Process
↓
MaterialSystem
↓
Device
↓
Property
```

というSchemaが有効です。

つまり、

```text
想定質問
↓
Schema設計
↓
Context Graph
↓
Graph RAG
```

という流れになります。

---



## Graph RAGの本質

RAGの本質は、

> 意味的に近い文章を検索すること

です。

Context Graphの本質は、

> 文書をNodeとEdgeで表現すること

です。

そしてGraph RAGの本質は、

> **質問をGraph Database上の探索問題へ変換し、Node・Edge・Pathを取得して回答すること**

です。

```text
論文
↓
Context Graph
↓
Graph Database
↓
Graph Query
↓
Evidence Path
↓
LLM
↓
Answer
```

これによりGraph RAGは、従来のRAGが苦手だった

* 因果関係の説明
* 論文横断推論
* Multi-hop推論
* 条件依存の知識探索

を実現できるようになります。




---



# Appendix: Context Graphでできること



## Context Graphでできること

Context Graphを利用すると、

```text
Strategy
↓
Process
↓
Material
↓
Property
```

という知識の流れを整理できます。

その結果、

* 著者が採用したStrategyを理解できる
* Strategyと結果の関係を追跡できる
* 論文間でStrategyを比較できる
* 同じStrategyが他論文でも有効かを調べられる
* 著者の主張が他論文と整合しているかを評価できる

ようになります。

つまり、

> 「著者が何を狙って、どのような結果を得たのか」

を構造的に理解できるようになります。

---



## 論文横断知識の統合

個々の論文は通常、不完全です。

例えば、

```text
論文A
A → B

論文B
B → C

論文C
C → D
```

のように知識が分散しています。

Context Graphを統合すると、

```text
A → B → C → D
```

という新たな知識パスが得られる場合があります。

つまり、

```text
論文A + 論文B + 論文C
```

を組み合わせることで、単独の論文では見えなかった知識を発見できる可能性があります。

これはGraph RAGの大きな利点の一つです。

---



## 知識補完（Knowledge Graph Completion）

論文から作成したContext Graphには欠落が存在することがよくあります。

例えば、

```text
Nb doping
↓
?
↓
Mobility increase
```

のように途中の機構が記載されていない場合があります。

多数の論文のContext Graphを統合すると、

```text
Nb doping
↓
Carrier concentration increase
↓
Mobility increase
```

という経路を推定できる可能性があります。

このような問題は、

> Knowledge Graph Completion

あるいは

> Link Prediction

と呼ばれます。

代表的な手法として、

* Matrix Factorization（SVDを含む）
* Node2Vec
* TransE
* ComplEx
* Graph Neural Network
* Knowledge Graph Embedding

などがあります。

したがって、

> 論文群から構築したContext Graphは、既存知識の整理だけでなく、欠落した関係や機構を推定するための基盤としても利用できる可能性があります。
